In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import os
import torch
import random
from torch.utils.data import Dataset, DataLoader

LABEL_NAMES = ['Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass','Nodule','Pneumonia','Pneumothorax','Consolidation','Edema','Emphysema','Fibrosis','Pleural_Thickening','Hernia']

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("? imports done")
print(f"Using seed: {SEED}")



? imports done
Using seed: 42


In [10]:
data = np.load("../data/chestmnist.npz")
print("Keys:", list(data.keys()))

Keys: ['train_images', 'val_images', 'test_images', 'train_labels', 'val_labels', 'test_labels']


In [11]:
X_train = data['train_images']
y_train = data['train_labels'].astype(np.float32)
X_val   = data['val_images']
y_val   = data['val_labels'].astype(np.float32)
X_test  = data['test_images']
y_test  = data['test_labels'].astype(np.float32)

print("Train:", X_train.shape, X_train.dtype)
print("Val:  ", X_val.shape, X_val.dtype)
print("Test: ", X_test.shape, X_test.dtype)

Train: (78468, 28, 28) uint8
Val:   (11219, 28, 28) uint8
Test:  (22433, 28, 28) uint8


In [12]:
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts

print("=== Class imbalance (train) ===")
for i, name in enumerate(LABEL_NAMES):
    print(f"{name:<20} pos={int(pos_counts[i]):>6}  neg={int(neg_counts[i]):>6}  ratio={neg_counts[i]/max(pos_counts[i], 1):.1f}x")

# Useful if you later use BCEWithLogitsLoss
pos_weight = torch.tensor(neg_counts / np.maximum(pos_counts, 1), dtype=torch.float32)
print("\nSuggested pos_weight for BCEWithLogitsLoss ready (shape):", tuple(pos_weight.shape))



=== Class imbalance (train) ===
Atelectasis          pos=  7996  neg= 70472  ratio=8.8x
Cardiomegaly         pos=  1950  neg= 76518  ratio=39.2x
Effusion             pos=  9261  neg= 69207  ratio=7.5x
Infiltration         pos= 13914  neg= 64554  ratio=4.6x
Mass                 pos=  3988  neg= 74480  ratio=18.7x
Nodule               pos=  4375  neg= 74093  ratio=16.9x
Pneumonia            pos=   978  neg= 77490  ratio=79.2x
Pneumothorax         pos=  3705  neg= 74763  ratio=20.2x
Consolidation        pos=  3263  neg= 75205  ratio=23.0x
Edema                pos=  1690  neg= 76778  ratio=45.4x
Emphysema            pos=  1799  neg= 76669  ratio=42.6x
Fibrosis             pos=  1158  neg= 77310  ratio=66.8x
Pleural_Thickening   pos=  2279  neg= 76189  ratio=33.4x
Hernia               pos=   144  neg= 78324  ratio=543.9x

Suggested pos_weight for BCEWithLogitsLoss ready (shape): (14,)


In [13]:
import sys
print(f"X_train size: {78468 * 224 * 224 * 4 / 1e9:.2f} GB")

X_train size: 15.75 GB


In [14]:
class ChestMNISTDataset(Dataset):
    def __init__(self, images, labels, mean=0.485, std=0.229):
        self.images = images  # uint8, stays in RAM cheap
        self.labels = labels
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].astype(np.float32) / 255.0
        img = (img - self.mean) / self.std  # backbone-style normalization
        img = torch.tensor(img).unsqueeze(0)               # (224, 224) ? (1, 224, 224)
        label = torch.tensor(self.labels[idx])
        return img, label

print("? Dataset class defined")



? Dataset class defined


In [15]:
train_dataset = ChestMNISTDataset(X_train, y_train)
val_dataset   = ChestMNISTDataset(X_val,   y_val)
test_dataset  = ChestMNISTDataset(X_test,  y_test)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, generator=generator)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")



Train batches: 2453
Val batches:   351
Test batches:  702


In [ ]:
imgs, labels = next(iter(train_loader))

print("Batch image shape:", imgs.shape)    # expect (32, 1, 224, 224)
print("Batch label shape:", labels.shape)  # expect (32, 14)
print("Pixel range:      ", imgs.min().item(), "-", imgs.max().item())  # expect 0.0 - 1.0
print("Label sample:\n", labels[0])        # binary vector of 14

In [ ]:
print("=== Dataset Sizes ===")
print(f"Train:      {len(train_dataset):>6} images | {len(train_loader):>4} batches")
print(f"Validation: {len(val_dataset):>6} images | {len(val_loader):>4} batches")
print(f"Test:       {len(test_dataset):>6} images | {len(test_loader):>4} batches")
print(f"Total:      {len(train_dataset)+len(val_dataset)+len(test_dataset):>6} images")
print(f"\nBatch size:  32")
print(f"Image shape: (1, 224, 224)")
print(f"Label shape: (14,) — multi-label binary")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(imgs[i].squeeze(), cmap='gray')
    active = [LABEL_NAMES[j] for j, v in enumerate(labels[i].int().tolist()) if v == 1]
    title = ', '.join(active) if active else 'No disease detected'
    ax.set_title(title, fontsize=8)
    ax.axis('off')
plt.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()



1. Imports
2. Load raw uint8 data
3. Exploratory analysis
4. ChestMNISTDataset class (lazy normalization + channel dim)
5. DataLoaders (batch=32, workers=2)
6. Sanity checks (shape, range, visuals)
7. Sizes summary

In [ ]:
# Quick imbalance experiment: BCE vs BCE + pos_weight
import copy

def build_tiny_model():
    return torch.nn.Sequential(
        torch.nn.Conv2d(1, 8, kernel_size=3, padding=1),
        torch.nn.ReLU(),
        torch.nn.MaxPool2d(2),
        torch.nn.Conv2d(8, 16, kernel_size=3, padding=1),
        torch.nn.ReLU(),
        torch.nn.AdaptiveAvgPool2d((1, 1)),
        torch.nn.Flatten(),
        torch.nn.Linear(16, len(LABEL_NAMES))
    )

def evaluate_multilabel(model, loader, device, max_batches=40, threshold=0.5):
    model.eval()
    tp = torch.zeros(len(LABEL_NAMES), device=device)
    fp = torch.zeros(len(LABEL_NAMES), device=device)
    fn = torch.zeros(len(LABEL_NAMES), device=device)

    with torch.no_grad():
        for b, (x, y) in enumerate(loader):
            if b >= max_batches:
                break
            x = x.to(device)
            y = y.to(device)
            p = (torch.sigmoid(model(x)) >= threshold).float()
            tp += (p * y).sum(0)
            fp += (p * (1 - y)).sum(0)
            fn += ((1 - p) * y).sum(0)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    macro = {
        'precision': precision.mean().item(),
        'recall': recall.mean().item(),
        'f1': f1.mean().item(),
    }
    return macro, precision.cpu(), recall.cpu(), f1.cpu()

def run_short_train(loss_fn, device, steps=120, lr=1e-3):
    torch.manual_seed(SEED)
    model = build_tiny_model().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()

    running_loss = 0.0
    step = 0
    while step < steps:
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()
            running_loss += loss.item()
            step += 1
            if step >= steps:
                break

    avg_loss = running_loss / steps
    macro, per_p, per_r, per_f1 = evaluate_multilabel(model, val_loader, device=device, max_batches=40)
    return avg_loss, macro, per_p, per_r, per_f1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Baseline
base_loss_fn = torch.nn.BCEWithLogitsLoss()
base_loss, base_macro, base_p, base_r, base_f1 = run_short_train(base_loss_fn, device=device)

# Weighted
weighted_loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
w_loss, w_macro, w_p, w_r, w_f1 = run_short_train(weighted_loss_fn, device=device)

print('\n=== Quick Comparison (same tiny model, short run) ===')
print(f"Baseline BCE       | loss={base_loss:.4f} | macro P={base_macro['precision']:.3f} R={base_macro['recall']:.3f} F1={base_macro['f1']:.3f}")
print(f"BCE + pos_weight   | loss={w_loss:.4f} | macro P={w_macro['precision']:.3f} R={w_macro['recall']:.3f} F1={w_macro['f1']:.3f}")

print('\nPer-class recall change (weighted - baseline):')
recall_delta = w_r - base_r
for i, name in enumerate(LABEL_NAMES):
    print(f"{name:<20} ?recall={recall_delta[i].item():+0.3f}  base={base_r[i].item():0.3f}  weighted={w_r[i].item():0.3f}")



In [ ]:
print(LABEL_NAMES)
print("num names:", len(LABEL_NAMES), "| y_train dims:", y_train.shape[1])
assert len(LABEL_NAMES) == y_train.shape[1]